# SCOMS — Frozen Source-Constrained Operating-Point Selection
**Scientia Iranica · Transactions D**. Four-corpus exploratory follow-up. Target labels are not used for within-run model selection, but earlier v1–v4 target results have influenced the research design; see `LOCKED_PROTOCOL_EN.md`. Real results are generated **only** by Run All on authentic source files.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('tldextract') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tldextract==5.3.0'])
print('Dependencies ready')

## Input files
Attach the same four datasets used in v4. The previously successful Kaggle paths are auto-used when they exist; otherwise the code scans exact filenames under `/kaggle/input`.

In [ ]:
# Dataset paths observed in the successful v4 run; use only if they still exist.
from pathlib import Path
KNOWN_INPUTS = {'PhiUSIIL': '/kaggle/input/datasets/alirezaesmaili71/legitphish/phiusiilphishingurldataset/PhiUSIIL_Phishing_URL_Dataset.csv', 'URL-Phish': '/kaggle/input/datasets/alirezaesmaili71/legitphish/URL-Phish A Feature-Engineered Dataset for Phishin (1)/URL-Phish A Feature-Engineered Dataset for Phishin/Dataset.csv', 'LegitPhish': '/kaggle/input/datasets/alirezaesmaili71/legitphish/LegitPhish Dataset (1)/LegitPhish Dataset/url_features_extracted1.csv', 'CompPhish': '/kaggle/input/datasets/alirezaesmaili71/all-features-threshold90/Mapping_File.xlsx'}
DATA_PATHS = {name:(path if Path(path).is_file() else None) for name,path in KNOWN_INPUTS.items()}
print('Existing known paths:',{name:path for name,path in DATA_PATHS.items() if path})
# If a path is None, the runner searches /kaggle/input by exact filename.
# You can override the mapping with full paths if discovery is ambiguous.


## Reproducible, self-contained experiment source
The complete code is embedded here. No separate Python file is required. Do not alter the frozen candidate grid or success criteria after seeing results.

In [ ]:
#!/usr/bin/env python3
"""SCOMS: source-constrained operational model selection, frozen evaluation.

Research prototype. Does not guarantee external FPR or establish novelty.
Labels are standardized to 1=phishing; no target labels/features used in fitting,
feature selection, tuning or threshold calibration.
"""
from __future__ import annotations
import os, re, io, json, csv, math, hashlib, zipfile, itertools, time, argparse
from pathlib import Path
from urllib.parse import urlsplit
from collections import Counter
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, beta
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix, average_precision_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CORPORA=['PhiUSIIL','URL-Phish','LegitPhish','CompPhish']
SEED=42
CAL_FPR=0.01
MAX_CLASS_SOURCE=12500
MAX_CLASS_DOMAIN=50
KS_SAMPLE=4000
MI_SAMPLE=25000
MIN_TRAIN_CLASS=80
MIN_EVAL_CLASS=100
MIN_EVAL_GROUPS=15
MIN_RETENTION=0.10
BOOT_REPS=400
K_VALUES=(8,16,24)
LAMBDA_VALUES=(0.0,0.1,0.3)
F_NAMES=['url_len','host_len','path_len','query_len','fragment_len','domain_dots',
         'subdomain_count','digit_count','digit_ratio','letter_ratio','hyphen_count',
         'dot_count','at_count','equal_count','question_count','amp_count',
         'percent_count','slash_count','underscore_count','tilde_count','colon_count',
         'hash_count','semicolon_count','is_https','is_ip','punycode','has_port',
         'url_entropy','host_entropy','token_count','host_hyphen_count']

DATA_PATTERNS={
 'PhiUSIIL':['PhiUSIIL_Phishing_URL_Dataset.csv','PhiUSIIL_source.zip','PhishRoute_results.zip'],
 'URL-Phish':['URL-Phish_Dataset.csv','URL-Phish.csv','URL-Phish_Dataset.zip','Dataset.csv'],
 'LegitPhish':['LegitPhish Dataset.csv','url_features_extracted1.csv','LegitPhish.csv'],
 'CompPhish':['Mapping_File.xlsx','Mapping_File.csv'],
}
SOURCE_BAD_LABEL={'PhiUSIIL':0,'URL-Phish':1,'LegitPhish':0,'CompPhish':1}


# Frozen operational choices; these are NOT optimized against an external target.
ALPHAS=(0.005,0.01,0.02,0.05,0.10)
RECALL_FLOOR=0.20
MIN_INNER_FOLDS=2
NONINFERIORITY_MARGIN=0.03
CCPS_FIXED_LAMBDA=0.10
CCPS_FIXED_K=16
BOOT_REPS=400
CANDIDATES=("LR_full","LR_prefix","HGB_full","HGB_prefix","LR_CCPS","HGB_CCPS")
FIXED_BASELINE_ALPHA=0.01

def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as fh:
  for buf in iter(lambda:fh.read(1024*1024),b''):h.update(buf)
 return h.hexdigest()


def candidates(roots):
 for root in roots:
  p=Path(root)
  if not p.exists():continue
  if p.is_file():yield p;continue
  for ext in ('*.csv','*.xlsx','*.zip'):
   yield from p.rglob(ext)


def locate(name, supplied, roots):
 if supplied:
  p=Path(supplied)
  if not p.is_file():raise FileNotFoundError(f'{name}: explicit input missing: {p}')
  return p
 allowed={s.lower() for s in DATA_PATTERNS[name]}
 hits=[]
 for p in candidates(roots):
  low=str(p).lower(); n=p.name.lower()
  if n not in allowed:continue
  if name=='URL-Phish' and n=='dataset.csv' and not any(s in low for s in ['url-phish','urlphish','65z9twcx3r']):continue
  hits.append(p)
 if not hits:raise FileNotFoundError(
  f'{name}: dataset not found. Add original dataset to Kaggle > Add Input; '
  f'accepted filenames: {DATA_PATTERNS[name]}. Explicit --{name.lower().replace("-","")} is supported.')
 hits.sort(key=lambda x:(len(str(x)),str(x)))
 if len(hits)>1:
  print(f'NOTE multiple possible {name} files:',[str(x) for x in hits[:5]],'\nChoosing',hits[0])
 return hits[0]


def read_raw(path,name):
 p=Path(path)
 if p.suffix.lower()=='.xlsx':
  for header in range(6):
   df=pd.read_excel(p,header=header,engine='openpyxl')
   c={re.sub(r'[^a-z0-9]+','',str(x).lower()) for x in df.columns}
   if any(x in c for x in ('url','websiteurl','weburl','link','webaddress')) and any(x in c for x in ('label','classlabel','class','target','status')):
    return df,str(p),sha256(p)
  raise ValueError(f'{name}: Mapping_File.xlsx does not expose recognizable URL+label columns. Inspect its header.')
 if p.suffix.lower()=='.csv':return pd.read_csv(p,low_memory=False),str(p),sha256(p)
 if p.suffix.lower()=='.zip':
  with zipfile.ZipFile(p) as z:
   want=[n for n in z.namelist() if n.lower().endswith('.csv') and 'phiusiil_phishing_url_dataset' in n.lower()]
   if name!='PhiUSIIL' or not want:raise ValueError(f'Unsupported ZIP content for {name}: {p}; extract original CSV/XLSX and attach it.')
   raw=z.read(want[0]);return pd.read_csv(io.BytesIO(raw),low_memory=False),f'{p}!{want[0]}',hashlib.sha256(raw).hexdigest()
 raise ValueError(f'Unexpected input type {p.suffix}')


def unify(df,name):
 columns={re.sub(r'[^a-z0-9]+','',str(c).strip().lower()):c for c in df.columns}
 u=next((columns[k] for k in ['url','websiteurl','weburl','link','webaddress','website','urladdress'] if k in columns),None)
 l=next((columns[k] for k in ['label','classlabel','target','class','status'] if k in columns),None)
 if not u or not l:raise ValueError(f'{name}: URL or label column not found. Available columns: {df.columns.tolist()}')
 part=df[[u,l]].copy();part.columns=['URL','source_label']
 part['URL']=part['URL'].astype('string').str.strip()
 labels=part.source_label.astype(str).str.strip().str.lower()
 # Explicitly map semantic string labels. Numbers follow the source's published convention.
 mapped=labels.map({'phishing':1,'phish':1,'malicious':1,'legitimate':0,'legit':0,'benign':0,'safe':0,'0':int(SOURCE_BAD_LABEL[name]==0),'1':int(SOURCE_BAD_LABEL[name]==1),'0.0':int(SOURCE_BAD_LABEL[name]==0),'1.0':int(SOURCE_BAD_LABEL[name]==1)})
 original=len(part)
 valid=part.URL.notna() & part.URL.str.len().gt(3) & mapped.notna()
 part=part[valid].copy();part['y']=mapped[valid].astype('int8');part=part.drop(columns=['source_label'])
 # URL string is left intact for lexical features; only trim whitespace.
 conflict=part.groupby('URL',sort=False).y.nunique()
 bad=set(conflict[conflict>1].index)
 n_bad=int(part.URL.isin(bad).sum());part=part[~part.URL.isin(bad)]
 part=part.drop_duplicates('URL').reset_index(drop=True)
 info={'corpus':name,'input_rows':int(original),'valid_rows':int(valid.sum()),'conflicting_label_rows':n_bad,'deduplicated_rows':int(len(part)), 'n_legitimate':int((part.y==0).sum()),'n_phishing':int((part.y==1).sum()),'url_column':str(u),'label_column':str(l)}
 if min(info['n_legitimate'],info['n_phishing'])<MIN_TRAIN_CLASS:raise ValueError(f'{name}: missing class or ambiguous labels: {info}')
 return part,info


def parse_url(raw):
 raw=str(raw).strip()
 if not re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://',raw): raw='http://'+raw
 try:
  p=urlsplit(raw); host=(p.hostname or '').lower().strip('.')
 except (ValueError,OverflowError):
  p=urlsplit('http://invalid.local/');host='invalid.local'
 return p,host


def shannon(s):
 if not s:return 0.
 c=Counter(s);n=len(s)
 return -sum((v/n)*math.log2(v/n) for v in c.values())


def feature_row(url,prefix_ablation=False):
 original=str(url).strip();p,host=parse_url(original)
 if prefix_ablation:
  # Remove scheme and leading www while retaining path/query so the feature budget
  # is identical. This is an ablation, not a claim of semantic equivalence.
  host=re.sub(r'^www\.','',host)
  original='http://'+host+(p.path or '')+(('?'+p.query) if p.query else '')+(('#'+p.fragment) if p.fragment else '')
  p,host=parse_url(original)
 raw=original.lower();base=raw.split('://',1)[-1];n=max(len(base),1)
 import ipaddress
 try:ipaddress.ip_address(host.strip('[]'));ip=1
 except ValueError:ip=0
 try:port=int(p.port is not None)
 except ValueError:port=0
 return [len(original),len(host),len(p.path),len(p.query),len(p.fragment),host.count('.'),max(0,len(host.split('.'))-2),
         sum(c.isdigit() for c in base),sum(c.isdigit() for c in base)/n,sum(c.isalpha() for c in base)/n,
         base.count('-'),base.count('.'),base.count('@'),base.count('='),base.count('?'),base.count('&'),
         base.count('%'),base.count('/'),base.count('_'),base.count('~'),base.count(':'),base.count('#'),
         base.count(';'),int(p.scheme.lower()=='https'),ip,int('xn--' in host),port,shannon(base),shannon(host),
         len([a for a in re.split(r'[^a-z0-9]+',base) if a]),host.count('-')]


def domain_of(host, extractor):
 if not host:return 'invalid.local'
 result=extractor(host)
 return (result.domain+'.'+result.suffix).lower() if result.suffix and result.domain else host


def extract_frame(frame,ext):
 n=len(frame);x=np.empty((n,len(F_NAMES)),dtype=np.float32);xp=np.empty_like(x);g=np.empty(n,dtype=object)
 for i,u in enumerate(frame.URL):
  x[i]=feature_row(u);xp[i]=feature_row(u,True)
  _,host=parse_url(u);g[i]=domain_of(host,ext)
  if (i+1)%100000==0:print('Parsed',i+1,'of',n,flush=True)
 assert np.isfinite(x).all() and np.isfinite(xp).all()
 return x,xp,g


def hash_fraction(s,seed):
 b=hashlib.sha256(f'{seed}|{s}'.encode('utf-8')).digest()
 return int.from_bytes(b[:8],'big')/(2**64)


def split_global(groups,seed=SEED,cal_fraction=.18):
 is_cal=np.fromiter((hash_fraction(str(x),seed)<cal_fraction for x in groups),dtype=bool,count=len(groups))
 return ~is_cal,is_cal


def cap_samples(d,seed=SEED):
 """Cap each class per domain and corpus; then cap total rows per class/corpus."""
 rng=np.random.default_rng(seed)
 sampled=[]
 for (name,y,dom), group in d.groupby(['corpus','y','domain'],sort=False):
  ix=group.index.to_numpy();sampled.extend(rng.choice(ix,size=min(len(ix),MAX_CLASS_DOMAIN),replace=False).tolist())
 out=d.loc[sampled].copy()
 selected=[]
 for (_, _),group in out.groupby(['corpus','y'],sort=False):
  ids=group.index.to_numpy();selected.extend(rng.choice(ids,size=min(len(ids),MAX_CLASS_SOURCE),replace=False).tolist())
 return out.loc[selected].sort_index().reset_index(drop=True)


def source_min_counts(d,origin_col='corpus'):
 if d.empty:return pd.DataFrame()
 rows=[]
 for name,part in d.groupby(origin_col):
  rows.append({'corpus':name,'n':len(part),'n_benign':int((part.y==0).sum()),'n_phishing':int((part.y==1).sum()),
               'groups_benign':int(part.loc[part.y==0,'domain'].nunique()),'groups_phishing':int(part.loc[part.y==1,'domain'].nunique())})
 return pd.DataFrame(rows)


def roc_direction(x,y):
 if len(np.unique(y))<2:return .5
 # AUC of raw feature. Rank-based test is direction-sensitive.
 if np.max(x)==np.min(x):return .5
 return float(roc_auc_score(y,x))


def stability_scores(d,X,seed=SEED,lambda_=0.1,kind='ccps'):
 """Return per-feature stable discrimination A, conditional source KS B and score.

 The sign is chosen by pooled training data ONLY. No target data used.
 Source class samples are bounded for computational stability.
 """
 nfeat=X.shape[1];rng=np.random.default_rng(seed)
 src=sorted(d.corpus.unique());y=d.y.to_numpy()
 pooled=np.array([roc_direction(X[:,j],y) for j in range(nfeat)])
 signs=np.where(pooled>=.5,1.,-1.)
 aucs=np.zeros((len(src),nfeat))
 for k,name in enumerate(src):
  idx=np.where(d.corpus.to_numpy()==name)[0]
  for j in range(nfeat):aucs[k,j]=roc_direction(X[idx,j],y[idx])
 a_min=np.maximum(0.,np.min((aucs-.5)*signs,axis=0))
 a_mean=np.maximum(0.,np.mean((aucs-.5)*signs,axis=0))
 B=np.zeros(nfeat)
 for yi in (0,1):
  arr={}
  for name in src:
   idx=np.where((d.corpus.to_numpy()==name)&(y==yi))[0]
   if len(idx)>KS_SAMPLE:idx=rng.choice(idx,size=KS_SAMPLE,replace=False)
   arr[name]=idx
  for i,left in enumerate(src):
   for right in src[i+1:]:
    if min(len(arr[left]),len(arr[right]))<20:continue
    for j in range(nfeat):
     kval=float(ks_2samp(X[arr[left],j],X[arr[right],j],method='asymp').statistic)
     B[j]=max(B[j],kval)
 if kind=='ccps':score=a_min-lambda_*B
 elif kind=='stability':score=a_min
 elif kind=='provenance':score=-B
 elif kind=='mean_ccps':score=a_mean-lambda_*B
 else:raise ValueError(kind)
 return {'score':score,'A_min':a_min,'A_mean':a_mean,'B':B,'sign':signs,'aucs':aucs,'sources':src}


def lr_fit(X,y):
 scaler=StandardScaler();a=scaler.fit_transform(X)
 model=LogisticRegression(max_iter=350,solver='liblinear',class_weight='balanced',random_state=SEED)
 model.fit(a,y)
 return scaler,model


def lr_predict(fit,X):return fit[1].predict_proba(fit[0].transform(X))[:,1]


def stats(y,score,th):
 y=np.asarray(y,dtype=np.int8);p=np.asarray(score,dtype=np.float64)
 yp=(p>=th).astype(np.int8)
 tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
 return {'n':len(y),'TP':int(tp),'FN':int(fn),'TN':int(tn),'FP':int(fp),
         'FPR':float(fp/(fp+tn)) if (fp+tn)>0 else None,
         'Recall':float(tp/(tp+fn)) if (tp+fn)>0 else None,
         'MCC':float(matthews_corrcoef(y,yp)) if len(y) else None,
         'Accuracy':float(np.mean(y==yp)) if len(y) else None,
         'PR_AUC':float(average_precision_score(y,p)) if np.unique(y).size==2 else None}


def exact_ci(k,n,alpha=.05):
 if n==0:return [None,None]
 return [float(beta.ppf(alpha/2,k,n-k+1)) if k else 0.,float(beta.ppf(1-alpha/2,k+1,n-k)) if k<n else 1.]


def load_all(pathdict,roots,ext):
 data={};rows=[];paths=[]
 for name in CORPORA:
  p=locate(name,pathdict.get(name),roots)
  raw,path,h=read_raw(p,name)
  base,info=unify(raw,name)
  x,xp,g=extract_frame(base,ext)
  invalid=(g=='invalid.local')
  info['invalid_hostname_removed']=int(invalid.sum())
  if invalid.any():
   base=base.loc[~invalid].reset_index(drop=True);x=x[~invalid];xp=xp[~invalid];g=g[~invalid]
  base['domain']=g;base['corpus']=name
  info.update({'distinct_domains':int(len(set(g))), 'domains_benign':int(len(set(g[base.y==0]))),'domains_phishing':int(len(set(g[base.y==1]))),'file':path,'sha256':h})
  data[name]={'frame':base,'raw':x,'prefix':xp};rows.append(info);paths.append(p)
  print(f'{name}: {len(base):,} records, {len(set(g)):,} distinct registered domains; labels {info["n_legitimate"]} benign/{info["n_phishing"]} phishing',flush=True)
 return data,rows,paths


def build_source(data,target,seed):
 sources=[s for s in CORPORA if s!=target]
 allframes=[]
 for source in sources:
  z=data[source]['frame'].copy()
  z['raw_index']=np.arange(len(z));allframes.append(z)
 frame=pd.concat(allframes,ignore_index=True)
 # conflicting labels between source corpora => all corresponding URLs removed
 conflicted=frame.groupby('URL').y.nunique()
 bad=set(conflicted[conflicted>1].index)
 conflictrows=int(frame.URL.isin(bad).sum())
 frame=frame[~frame.URL.isin(bad)].drop_duplicates('URL',keep='first').copy()
 frame=cap_samples(frame,seed)
 istrain,iscal=split_global(frame.domain,seed)
 tr=frame[istrain].reset_index(drop=True);cal=frame[iscal].reset_index(drop=True)
 # hard check: no domain group may cross train/cal under any corpus
 assert not (set(tr.domain)&set(cal.domain)),'GLOBAL_TRAIN_CAL_GROUP_LEAK'
 def getX(part,view):
  return np.vstack([data[s][view][ix] for s,ix in zip(part.corpus,part.raw_index)]).astype(np.float32) if len(part) else np.empty((0,len(F_NAMES)),dtype=np.float32)
 return tr,cal,getX(tr,'raw'),getX(cal,'raw'),getX(tr,'prefix'),getX(cal,'prefix'),{'source_conflict_rows_removed':conflictrows,'source_post_conflict_dedup_and_cap_removed':int(sum(len(data[s]['frame']) for s in sources)-conflictrows-len(frame)), 'train_count':len(tr),'cal_count':len(cal)}


def assess_quality(frame,original):
 n0=int((original.y==0).sum());n1=int((original.y==1).sum())
 if len(frame)==0:return {'n':0,'quality_primary':False,'retention_benign':0,'retention_phishing':0}
 t0=int((frame.y==0).sum());t1=int((frame.y==1).sum())
 g0=int(frame.loc[frame.y==0,'domain'].nunique());g1=int(frame.loc[frame.y==1,'domain'].nunique())
 r0=t0/max(n0,1);r1=t1/max(n1,1)
 return {'n':len(frame),'n_benign':t0,'n_phishing':t1,'n_groups_benign':g0,'n_groups_phishing':g1,'retention_benign':r0,'retention_phishing':r1,
  'quality_primary':bool(t0>=MIN_EVAL_CLASS and t1>=MIN_EVAL_CLASS and min(g0,g1)>=MIN_EVAL_GROUPS and min(r0,r1)>=MIN_RETENTION)}


# ---------- Frozen final source-constrained study ----------

def fit_model(name, xraw, xprefix, y, ccps_ix, seed):
    prefix = name.endswith('prefix')
    ix = ccps_ix if 'CCPS' in name else np.arange(xraw.shape[1])
    x = xprefix if prefix else xraw
    if name.startswith('LR_'):
        return {'name':name,'fit':lr_fit(x[:,ix],y),'ix':ix,'prefix':prefix,'family':'LR'}
    clf=HistGradientBoostingClassifier(max_iter=115,max_leaf_nodes=15,
                learning_rate=.08,l2_regularization=1.0,min_samples_leaf=45,random_state=seed)
    clf.fit(x[:,ix],y,sample_weight=compute_sample_weight('balanced',y))
    return {'name':name,'fit':clf,'ix':ix,'prefix':prefix,'family':'HGB'}

def predict_model(model,xraw,xprefix):
    x=(xprefix if model['prefix'] else xraw)[:,model['ix']]
    if model['family']=='LR':return lr_predict(model['fit'],x)
    return model['fit'].predict_proba(x)[:,1]

def exact_source_threshold(scores, calibration, alpha, min_benign=50):
    """Smallest score threshold satisfying per-source empirical FPR <= alpha.

    Predict phishing if score>=threshold. For each source a cutoff just beyond
    the (n - floor(alpha*n) - 1)-th sorted benign score ensures no more than
    floor(alpha*n) false positives, including with ties.
    """
    p=np.asarray(scores,dtype=float)
    th=[]; details=[]
    for src in sorted(calibration.corpus.unique()):
        mask=((calibration.corpus==src)&(calibration.y==0)).to_numpy()
        s=np.sort(p[mask]);n=len(s)
        if n < min_benign:
            details.append({'source':src,'n_benign':n,'status':'too_few_benign'});continue
        allowed=int(math.floor(alpha*n))
        if allowed>=n:t=-np.inf
        else:t=float(np.nextafter(s[n-allowed-1],np.inf))
        actual=float(np.mean(s>=t))
        assert actual<=alpha+1e-12,(src,actual,alpha)
        th.append(t)
        details.append({'source':src,'n_benign':n,'cutoff':t,'cal_FPR':actual,'status':'ok'})
    if len(th)<1:raise ValueError('No calibration source with enough benign rows.')
    threshold=float(max(th))
    return threshold,details

def ccps_indices(frame,xraw,seed):
    # Frozen negative-control subset; NOT the target-tuned v4 setting.
    if len(frame.corpus.unique())<2:return np.arange(min(CCPS_FIXED_K,xraw.shape[1]))
    info=stability_scores(frame.reset_index(drop=True),xraw,seed=seed,
                 lambda_=CCPS_FIXED_LAMBDA,kind='ccps')
    return np.argsort(-info['score'],kind='stable')[:CCPS_FIXED_K]

def collect_inner_folds(train,cal,xtr,xcal,xtrp,xcalp,seed):
    """Inner held-source folds. Training and calibration domains cannot enter held validation.

    Same capped data, source-only. Held-source rows used only to measure candidate
    operating points, never to fit their own feature selector or threshold.
    """
    srcs=sorted(train.corpus.unique())
    folds=[]; manifest=[]
    for held in srcs:
        other=[x for x in srcs if x!=held]
        # Exclude registered domains appearing in ANY other training source,
        # including its calibration partition, to prevent cross-source memorization.
        blocked=set(train.loc[train.corpus.isin(other),'domain']) | set(cal.loc[cal.corpus.isin(other),'domain'])
        a=(train.corpus.isin(other)).to_numpy()
        c=(cal.corpus.isin(other)).to_numpy()
        h0=(train.corpus==held).to_numpy()
        h=h0 & ~train.domain.isin(blocked).to_numpy()
        ha=train[h];oa=train[a];ca=cal[c]
        assert not (set(ha.domain) & (set(oa.domain) | set(ca.domain))), 'INNER_SOURCE_DOMAIN_LEAK'
        assert not (set(oa.domain) & set(ca.domain)), 'INNER_TRAIN_CAL_DOMAIN_LEAK'
        old_count=train[h0]
        q=assess_quality(ha,old_count)
        eligible=(q['quality_primary'] and oa.y.nunique()==2 and ca.y.nunique()==2 and
                all(int((ca[ca.corpus==s].y==0).sum())>=50 for s in other) and
                all(int((oa[oa.corpus==s].y==0).sum())>=50 for s in other))
        manifest.append({'held_source':held,'training_sources':'|'.join(other),
                         'n_before_domain_filter':int(h0.sum()),**q,'eligible_inner':bool(eligible)})
        if eligible:
            folds.append({'held':held,'train':oa.reset_index(drop=True),
                     'cal':ca.reset_index(drop=True),'val':ha.reset_index(drop=True),
                     'xt':xtr[a],'xc':xcal[c],'xh':xtr[h],
                     'xtp':xtrp[a],'xcp':xcalp[c],'xhp':xtrp[h]})
    return folds,manifest

def select_from_sources(train,cal,xtr,xcal,xtrp,xcalp,seed):
    folds,manifest=collect_inner_folds(train,cal,xtr,xcal,xtrp,xcalp,seed)
    assessments=[]; errors=[]; calibrated={}
    for fold in folds:
        held=fold['held']; ccps=ccps_indices(fold['train'],fold['xt'],seed)
        for name in CANDIDATES:
            try:
                model=fit_model(name,fold['xt'],fold['xtp'],fold['train'].y.to_numpy(),ccps,seed)
                sc=predict_model(model,fold['xc'],fold['xcp'])
                sv=predict_model(model,fold['xh'],fold['xhp'])
                calibrated[(held,name)]={'prob_cal':sc,'prob_val':sv}
                for alpha in ALPHAS:
                    th,per=exact_source_threshold(sc,fold['cal'],alpha)
                    met=stats(fold['val'].y.to_numpy(),sv,th)
                    assessments.append({'held_source':held,'model':name,'alpha':alpha,'threshold':th,
                                        'cal_source_details':json.dumps(per),**met})
            except Exception as e:
                errors.append({'held_source':held,'model':name,'error':str(e)})
    d=pd.DataFrame(assessments)
    ranked=[]
    for name,alpha in itertools.product(CANDIDATES,ALPHAS):
        sub=d[(d.model==name)&(np.isclose(d.alpha,alpha))] if len(d) else pd.DataFrame()
        vals={'model':name,'alpha':alpha,'eligible_inner_folds':len(folds),
              'successful_inner_folds':len(sub),'feasible':False,
              'worst_FPR':None,'worst_Recall':None,'mean_MCC':None}
        if len(sub)==len(folds) and len(folds)>=MIN_INNER_FOLDS:
            vals.update({'worst_FPR':float(sub.FPR.max()),'worst_Recall':float(sub.Recall.min()),
                         'mean_MCC':float(sub.MCC.mean()),
                         'feasible':bool((sub.Recall>=RECALL_FLOOR).all())})
        ranked.append(vals)
    feasible=[r for r in ranked if r['feasible']]
    # Prespecified tie breaks: min worst FPR, max worst recall, max mean MCC,
    # simpler architecture and smaller alpha. No target rows are involved.
    if feasible:
        complexity={n:i for i,n in enumerate(CANDIDATES)}
        winner=min(feasible,key=lambda a:(a['worst_FPR'],-a['worst_Recall'],-a['mean_MCC'],
                        complexity[a['model']],a['alpha']))
        decision={'selection_status':'FEASIBLE_SOURCE_ONLY', 'selected_model':winner['model'],
                 'selected_alpha':winner['alpha'],'n_eligible_inner':len(folds),
                 'selected_worst_inner_FPR':winner['worst_FPR'],
                 'selected_worst_inner_Recall':winner['worst_Recall'],
                 'fallback_used':False}
    else:
        # No feasible solution: the fixed baseline is a *fallback*, not a winner.
        decision={'selection_status':'INFEASIBLE_FALLBACK_BASELINE',
                 'selected_model':'LR_prefix','selected_alpha':FIXED_BASELINE_ALPHA,
                 'n_eligible_inner':len(folds),'selected_worst_inner_FPR':None,
                 'selected_worst_inner_Recall':None,'fallback_used':True}
    return decision,manifest,assessments,ranked,errors

def bootstrap_delta(y,groups,p_new,p_ref,seed,reps=BOOT_REPS):
    # Binary prediction indicators; bootstrap the same domain clusters for both.
    y=np.asarray(y,dtype=np.int8);p_new=np.asarray(p_new,dtype=bool)
    p_ref=np.asarray(p_ref,dtype=bool);codes,unique=pd.factorize(groups)
    count=len(unique)
    if count<MIN_EVAL_GROUPS:return {'valid':False,'reason':'insufficient domain clusters'}
    binc=lambda z:np.bincount(codes,weights=z.astype(np.int32),minlength=count)
    neg=binc(y==0);pos=binc(y==1)
    a_fp=binc((y==0)&p_new);b_fp=binc((y==0)&p_ref)
    a_tp=binc((y==1)&p_new);b_tp=binc((y==1)&p_ref)
    rng=np.random.default_rng(seed);diff_fp=[];diff_recall=[]
    for _ in range(reps):
        idx=rng.integers(0,count,count)
        nneg=neg[idx].sum();npos=pos[idx].sum()
        if min(nneg,npos)==0:continue
        diff_fp.append(float((a_fp[idx].sum()-b_fp[idx].sum())/nneg))
        diff_recall.append(float((a_tp[idx].sum()-b_tp[idx].sum())/npos))
    if len(diff_fp)<int(reps*.9):return {'valid':False,'reason':'too many single-class bootstrap draws'}
    d_fpr=list(map(float,np.quantile(diff_fp,[.025,.5,.975])))
    d_rec=list(map(float,np.quantile(diff_recall,[.025,.5,.975])))
    return {'valid':True,'n_groups':count,'replicates':len(diff_fp),
            'delta_FPR_new_minus_ref_CI95':d_fpr,
            'delta_Recall_new_minus_ref_CI95':d_rec,
            'fpr_improvement_supported':bool(d_fpr[2]<0),
            'recall_noninferiority_supported':bool(d_rec[0]>=-NONINFERIORITY_MARGIN),
            'predeclared_paired_success':bool(d_fpr[2]<0 and d_rec[0]>=-NONINFERIORITY_MARGIN)}

def hmac_id(text_value,secret):
    # Hash with undisclosed in-memory key to avoid publishing guessable bare URL SHA256.
    return hashlib.blake2b(str(text_value).encode('utf8'),key=secret,digest_size=16).hexdigest()

def save_target_predictions(out,target,df,strict,prob,thresholds,secret):
    # Strict-unseen sample identifiers + binary outcomes & rounded scores.
    # Raw URLs and registered domains never leave the Kaggle session.
    ids=np.flatnonzero(strict)
    base=pd.DataFrame({'id_hash':[hmac_id(v,secret) for v in df.URL.iloc[ids]],
          'domain_hash':[hmac_id(v,secret) for v in df.domain.iloc[ids]],
          'y':df.y.iloc[ids].to_numpy()})
    for name,sc in prob.items():
        t=thresholds[name]
        base[name+'_score']=np.around(sc[ids],7)
        base[name+'_predict']=(sc[ids]>=t).astype('int8')
    fname=out/f'Hashed_Predictions_{target}.csv.gz'
    base.to_csv(fname,index=False,compression='gzip')
    groupcols=[c for c in base.columns if c.endswith('_predict')]
    # Vectorized aggregation; iterating in Python over ~100k domains is too slow.
    summary=pd.DataFrame({'domain_hash':base.domain_hash,
            'n_benign':(base.y==0).astype('int8'),
            'n_phishing':(base.y==1).astype('int8')})
    for col in groupcols:
        summary[col+'_FP']=((base.y==0)&(base[col]==1)).astype('int8')
        summary[col+'_TP']=((base.y==1)&(base[col]==1)).astype('int8')
    grouped=summary.groupby('domain_hash',sort=False,as_index=False).sum()
    grouped.to_csv(out/f'Domain_Confusions_{target}.csv.gz',index=False,compression='gzip')
    return {'n_hashed_records':len(base),'n_hashed_domains':len(grouped)}

def evaluate_outer(data,out,seed):
    records=[];quality=[];inner_rows=[];inner_rank=[];inner_manifest=[];decisions=[];calibration=[]
    boot_rows=[];selection_audit=[]
    secret=os.urandom(32)  # not exported, to protect URL membership privacy
    for target in CORPORA:
        print('\n'+'='*60+'\nHELD-OUT:',target,flush=True)
        tr,cal,xtr,xcal,xtrp,xcalp,audit=build_source(data,target,seed)
        choice,fold_manifest,inners,ranks,errors=select_from_sources(tr,cal,xtr,xcal,xtrp,xcalp,seed)
        decisions.append({'target':target,**choice,**audit})
        inner_manifest.extend([{'target':target,**r} for r in fold_manifest]);inner_rows.extend([{'target':target,**r} for r in inners])
        inner_rank.extend([{'target':target,**r} for r in ranks])
        selection_audit.extend([{'target':target,**r} for r in errors])
        print('PRE-TEST source-only decision',choice,flush=True)
        subset=ccps_indices(tr,xtr,seed)
        models={};scores={};thresholds={}
        for name in CANDIDATES:
            mdl=fit_model(name,xtr,xtrp,tr.y.to_numpy(),subset,seed)
            models[name]=mdl
            cc=predict_model(mdl,xcal,xcalp)
            tg=data[target]
            scores[name]=predict_model(mdl,tg['raw'],tg['prefix'])
            th,per=exact_source_threshold(cc,cal,FIXED_BASELINE_ALPHA)
            thresholds[name]=th
            calibration.append({'target':target,'model':name,'alpha':FIXED_BASELINE_ALPHA,
                                'threshold':th,'details_json':json.dumps(per)})
        selected_name=choice['selected_model'];selected_alpha=float(choice['selected_alpha'])
        if not math.isclose(selected_alpha,FIXED_BASELINE_ALPHA):
            cal_scores=predict_model(models[selected_name],xcal,xcalp)
            selected_th,per=exact_source_threshold(cal_scores,cal,selected_alpha)
            calibration.append({'target':target,'model':'SCOMS_selected','base_model':selected_name,
                                'alpha':selected_alpha,'threshold':selected_th,
                                'details_json':json.dumps(per)})
        else:selected_th=thresholds[selected_name]
        scores['SCOMS_selected']=scores[selected_name]
        thresholds['SCOMS_selected']=selected_th
        df=data[target]['frame'];shared=set()
        for corpus in CORPORA:
            if corpus!=target:shared.update(data[corpus]['frame'].domain.to_numpy())
        strict=(~df.domain.isin(shared)).to_numpy()
        for policy,mask in [('full_external',np.ones(len(df),dtype=bool)),('strict_unseen',strict)]:
            q=assess_quality(df[mask],df)
            quality.append({'target':target,'policy':policy,**q})
            for name,probs in scores.items():
                met=stats(df.y.to_numpy()[mask],probs[mask],thresholds[name])
                records.append({'target':target,'policy':policy,'model':name,
                         'chosen_base_model':selected_name if name=='SCOMS_selected' else name,
                         'selection_status':choice['selection_status'] if name=='SCOMS_selected' else 'FROZEN_BASELINE',
                         'cal_alpha':selected_alpha if name=='SCOMS_selected' else FIXED_BASELINE_ALPHA,
                         'threshold':thresholds[name],**q,**met,
                         'alerts_per_10000_benign':met['FPR']*10000 if met['FPR'] is not None else None,
                         'FPR_ClopperPearson95':json.dumps(exact_ci(met['FP'],met['FP']+met['TN'])),
                         'Recall_ClopperPearson95':json.dumps(exact_ci(met['TP'],met['TP']+met['FN']))})
        # Critical: do not bootstrap invalid strict subsets or report them as primary.
        q=assess_quality(df[strict],df)
        if q['quality_primary']:
            y=df.y.to_numpy()[strict];g=df.domain.to_numpy()[strict]
            chosen=(scores['SCOMS_selected'][strict]>=thresholds['SCOMS_selected'])
            for baseline in ('LR_prefix','HGB_full'):
                ref=(scores[baseline][strict]>=thresholds[baseline])
                bs=bootstrap_delta(y,g,chosen,ref,seed+37+CORPORA.index(target))
                boot_rows.append({'target':target,'reference':baseline,'strict_primary':True,
                                  'selected_was_feasible':not choice['fallback_used'],**bs})
        else:boot_rows.append({'target':target,'reference':'N/A','strict_primary':False,
                               'valid':False,'reason':'class-retention gate failure'})
        save_target_predictions(out,target,df,strict,scores,thresholds,secret)
        print(target,'strict n=',int(strict.sum()),'quality=',q['quality_primary'],flush=True)
    tables={
        'Table_ModelSelection.csv':pd.DataFrame(decisions),
        'Table_Inner_Source_Folds.csv':pd.DataFrame(inner_manifest),
        'Table_Inner_Candidate_Performance.csv':pd.DataFrame(inner_rows),
        'Table_Inner_Candidate_Ranking.csv':pd.DataFrame(inner_rank),
        'Table_Outer_Calibration.csv':pd.DataFrame(calibration),
        'Table_Outer_Performance.csv':pd.DataFrame(records),
        'Table_Class_Retention.csv':pd.DataFrame(quality),
    }
    for name,df in tables.items():df.to_csv(out/name,index=False)
    pd.DataFrame(selection_audit).to_csv(out/'Table_Selection_Errors.csv',index=False)
    (out/'Paired_Cluster_Bootstrap.json').write_text(json.dumps(boot_rows,indent=2),encoding='utf-8')
    return records,quality,decisions,boot_rows

def decide(out,perf,quality,decisions,bootstrap):
    p=pd.DataFrame(perf);q=pd.DataFrame(quality)
    included=q[(q.policy=='strict_unseen')&(q.quality_primary==True)].target.tolist()
    per=[]
    for target in included:
        selected=next(d for d in decisions if d['target']==target)
        comparisons=[]
        for ref in ['LR_prefix','HGB_full']:
            bs=next((r for r in bootstrap if r.get('target')==target and r.get('reference')==ref),None)
            comparisons.append({'reference':ref,'success':bool(bs and bs.get('predeclared_paired_success',False)),
                                'bootstrap_valid':bool(bs and bs.get('valid',False))})
        # Unqualified fallback cannot be credited to success, even if equal baseline
        success=not selected['fallback_used'] and all(x['success'] for x in comparisons)
        per.append({'target':target,'source_only_feasible':not selected['fallback_used'],
                    'success_vs_both_fixed_baselines':success,'comparisons':comparisons})
    successes=sum(x['success_vs_both_fixed_baselines'] for x in per)
    gate={'status':'COMPLETED_NOT_A_JOURNAL_ACCEPTANCE_PREDICTION',
       'study_type':'exploratory_frozen_follow_up_after_v4_target_exposure',
       'target_label_peeking_this_run':'none',
       'important_adaptivity_caveat':'The four target datasets have already influenced v1-v4 design decisions; v5 is not a pristine independent prospective confirmation.',
       'strict_eligible_targets':included,'n_strict_eligible_targets':len(included),
       'targets':per,'successes':successes,'predeclared_required_successes':2,
       'gate_passed':bool(len(included)>=3 and successes>=2),
       'interpretation':'Experimental method-selection advantage requires novelty and independent replication; pass/fail does not imply journal acceptance.'}
    (out/'SCIENTIFIC_GATE.json').write_text(json.dumps(gate,indent=2),encoding='utf-8')
    with open(out/'REPORT_FA.md','w',encoding='utf-8') as f:
        f.write('# ارزیابی نهایی SCOMS برای Scientia Iranica\n\n')
        f.write('**توجه: این مطالعه بعد از مشاهده نتایج v1-v4 طراحی شده و تأیید مستقل آینده‌نگر نیست.**\\n\\n'.replace('\\n','\n'))
        f.write('تنها داده‌های سه منبع آموزشی برای انتخاب پیکربندی و آستانه به کار رفته‌اند. درون هر تکرار، مجموعه هدف صرفاً برای ارزیابی است.\n\n')
        f.write('## تصمیم مستقل از هدف\n\n```\n'+pd.DataFrame(decisions)[['target','selection_status','selected_model','selected_alpha','n_eligible_inner']].to_string(index=False)+'\n```\n\n')
        f.write('## نتایج اصلی\n\n')
        valid=p[(p.policy=='strict_unseen')&(p.quality_primary==True)]
        f.write('```\n'+valid[['target','model','FPR','Recall','MCC','PR_AUC','alerts_per_10000_benign']].to_string(index=False)+'\n```\n\n')
        f.write('## نتیجه معیار توقف\n\n```json\n'+json.dumps(gate,ensure_ascii=False,indent=2)+'\n```\n')
        f.write('\nاین آزمون تضمین FPR در دنیای واقعی یا تأیید نوآوری نظری نیست. در صورت شکست، جست‌وجوی مدل جدید متوقف شود.\n')
    return gate

def charts(out):
    p=pd.read_csv(out/'Table_Outer_Performance.csv');d=p[(p.policy=='strict_unseen')&(p.quality_primary==True)]
    if d.empty:return
    use=['SCOMS_selected','LR_prefix','HGB_full','LR_full','HGB_prefix','LR_CCPS','HGB_CCPS']
    for metric,filename in [('FPR','Figure_Strict_FPR.png'),('Recall','Figure_Strict_Recall.png'),('MCC','Figure_Strict_MCC.png')]:
        mat=d[d.model.isin(use)].pivot(index='target',columns='model',values=metric)
        mat=mat.reindex(index=[s for s in CORPORA if s in mat.index],columns=use)
        ax=mat.plot(kind='bar',figsize=(11,5),edgecolor='black',linewidth=.35,rot=0)
        ax.set_ylabel(metric);ax.set_xlabel('');ax.grid(axis='y',alpha=.25)
        if metric!='MCC':ax.set_ylim(bottom=0)
        ax.legend(loc='best',fontsize=8,ncol=2)
        ax.figure.tight_layout();ax.figure.savefig(out/filename,dpi=300,bbox_inches='tight')
        plt.close(ax.figure)

def run_scoms(args):
    start=time.time();out=Path(args.out).resolve();out.mkdir(parents=True,exist_ok=True)
    try:import tldextract
    except ImportError as e:raise RuntimeError('The package tldextract is required. Enable Kaggle Internet and pip install tldextract==5.3.0; never approximate registered domains by split(".").') from e
    ext=tldextract.TLDExtract(suffix_list_urls=(),cache_dir=str(out/'tld_cache'))
    roots=args.search_roots.split(';') if args.search_roots else ['/kaggle/input','/kaggle/working']
    pathdict={'PhiUSIIL':args.phiusiil,'URL-Phish':args.urlphish,'LegitPhish':args.legitphish,'CompPhish':args.compphish}
    data,manifest,files=load_all(pathdict,roots,ext)
    pd.DataFrame(manifest).to_csv(out/'Dataset_Manifest.csv',index=False)
    rows=[]
    for a,b in itertools.combinations(CORPORA,2):
        u=data[a]['frame'][['URL','y']].merge(data[b]['frame'][['URL','y']],on='URL',suffixes=('_a','_b'))
        rows.append({'a':a,'b':b,'same_url_count':len(u),'label_conflicts':int((u.y_a!=u.y_b).sum()),
                     'shared_registered_domains':len(set(data[a]['frame'].domain)&set(data[b]['frame'].domain))})
    pd.DataFrame(rows).to_csv(out/'Corpus_Overlap_Audit.csv',index=False)
    records,quality,decisions,bs=evaluate_outer(data,out,args.seed)
    gate=decide(out,records,quality,decisions,bs)
    charts(out)
    runmeta={'version':'SCOMS_FINAL_V5','time_seconds':round(time.time()-start,2),
       'params':{'seed':args.seed,'alphas':ALPHAS,'recall_floor':RECALL_FLOOR,'min_inner_folds':MIN_INNER_FOLDS,
       'ccps_lambda':CCPS_FIXED_LAMBDA,'ccps_k':CCPS_FIXED_K,'bootstrap_reps':BOOT_REPS,
       'min_retention_each_class':MIN_RETENTION,'fixed_alpha':FIXED_BASELINE_ALPHA,
       'noninferiority_margin':NONINFERIORITY_MARGIN},
       'libraries':{'numpy':np.__version__,'pandas':pd.__version__,
                    'sklearn':__import__('sklearn').__version__,
                    'tldextract':getattr(tldextract,'__version__','unknown')},
       'manifest':manifest,'gate':gate}
    (out/'RUN_MANIFEST.json').write_text(json.dumps(runmeta,indent=2,default=str),encoding='utf-8')
    (out/'RUN_STATUS.txt').write_text('EXECUTED_SCOMS_FOUR_CORPUS\n'+str(round(time.time()-start,2))+' seconds\n',encoding='utf-8')
    output=out.parent/'SCOMS_ScientiaIranica_FinalResults.zip'
    with zipfile.ZipFile(output,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for f in out.rglob('*'):
            if f.is_file() and 'tld_cache' not in f.parts:z.write(f,arcname=f.relative_to(out))
    print('\nResults archive:',output,'\nScientific gate:',gate,flush=True)
    return gate


## Run all predeclared folds once
This can be computationally intensive. Do not use target-corpus performance for selecting candidates.

In [ ]:
# Frozen, complete run; executes the 4 leave-one-corpus-out folds.
import argparse
run_scoms(argparse.Namespace(
    phiusiil=DATA_PATHS['PhiUSIIL'],
    urlphish=DATA_PATHS['URL-Phish'],
    legitphish=DATA_PATHS['LegitPhish'],
    compphish=DATA_PATHS['CompPhish'],
    search_roots='/kaggle/input;/kaggle/working',
    seed=42,
    out='/kaggle/working/SCOMS_FinalOutput'
))


In [ ]:
from pathlib import Path
import pandas as pd, json
out=Path('/kaggle/working/SCOMS_FinalOutput')
print('SCIENTIFIC GATE (not a publication decision):')
print(json.dumps(json.loads((out/'SCIENTIFIC_GATE.json').read_text()),indent=2))
print('
Selected models and fallback status:')
display(pd.read_csv(out/'Table_ModelSelection.csv')[['target','selection_status','selected_model','selected_alpha','n_eligible_inner']])
print('
Main primary results:')
p=pd.read_csv(out/'Table_Outer_Performance.csv')
display(p[(p.policy=='strict_unseen') & (p.quality_primary==True)][['target','model','FPR','Recall','MCC','PR_AUC','alerts_per_10000_benign']])
print('
Download from Kaggle Output: /kaggle/working/SCOMS_ScientiaIranica_FinalResults.zip')
